In [ ]:
from collections.abc import Sequence
from sklearn import preprocessing
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shutil
import os


# Encode text values to dummy variables(i.e. [1,0,0],[0,1,0],[0,0,1] for red,green,blue)
def encode_text_dummy(df, name):
    dummies = pd.get_dummies(df[name])
    for x in dummies.columns:
        dummy_name = "{}-{}".format(name, x)
        df[dummy_name] = dummies[x]
    df.drop(name, axis=1, inplace=True)


# Encode text values to indexes(i.e. [1],[2],[3] for red,green,blue).
def encode_text_index(df, name):
    le = preprocessing.LabelEncoder()
    df[name] = le.fit_transform(df[name])
    return le.classes_


# Encode a numeric column as zscores
def encode_numeric_zscore(df, name, mean=None, sd=None):
    if mean is None:
        mean = df[name].mean()

    if sd is None:
        sd = df[name].std()

    df[name] = (df[name] - mean) / sd


# Convert all missing values in the specified column to the median
def missing_median(df, name):
    med = df[name].median()
    df[name] = df[name].fillna(med)


# Convert all missing values in the specified column to the default
def missing_default(df, name, default_value):
    df[name] = df[name].fillna(default_value)


# Convert a Pandas dataframe to the x,y inputs that TensorFlow needs
def to_xy(df, target):
    result = []
    for x in df.columns:
        if x != target:
            result.append(x)
    # find out the type of the target column. 
    target_type = df[target].dtypes
    target_type = target_type[0] if isinstance(target_type, Sequence) else target_type
    # Encode to int for classification, float otherwise. TensorFlow likes 32 bits.
    if target_type in (np.int64, np.int32):
        # Classification
        dummies = pd.get_dummies(df[target])
        return df[result].values.astype(np.float32), dummies.values.astype(np.float32)
    else:
        # Regression
        return df[result].values.astype(np.float32), df[target].values.astype(np.float32)

# Nicely formatted time string
def hms_string(sec_elapsed):
    h = int(sec_elapsed / (60 * 60))
    m = int((sec_elapsed % (60 * 60)) / 60)
    s = sec_elapsed % 60
    return "{}:{:>02}:{:>05.2f}".format(h, m, s)


# Regression chart.
def chart_regression(pred,y,sort=True):
    t = pd.DataFrame({'pred' : pred, 'y' : y.flatten()})
    if sort:
        t.sort_values(by=['y'],inplace=True)
    a = plt.plot(t['y'].tolist(),label='expected')
    b = plt.plot(t['pred'].tolist(),label='prediction')
    plt.ylabel('output')
    plt.legend()
    plt.show()

# Remove all rows where the specified column is +/- sd standard deviations
def remove_outliers(df, name, sd):
    drop_rows = df.index[(np.abs(df[name] - df[name].mean()) >= (sd * df[name].std()))]
    df.drop(drop_rows, axis=0, inplace=True)


# Encode a column to a range between normalized_low and normalized_high.
def encode_numeric_range(df, name, normalized_low=-1, normalized_high=1,
                         data_low=None, data_high=None):
    if data_low is None:
        data_low = min(df[name])
        data_high = max(df[name])

    df[name] = ((df[name] - data_low) / (data_high - data_low)) \
               * (normalized_high - normalized_low) + normalized_low


In [ ]:
features_df = pd.read_csv('dataset/NUSW-NB15_features.csv', encoding="ISO-8859-1")
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
features_df

In [ ]:
train_df = pd.read_csv('dataset/UNSW_NB15_training-set.csv', na_values='-')
test_df = pd.read_csv('dataset/UNSW_NB15_test-set.csv', na_values='-')


In [ ]:
train_df.head()

In [ ]:
train_df.columns

In [ ]:
test_df.head()

In [ ]:
print(f'train shape: {train_df.shape}')
print(f'test shape: {test_df.shape}')

In [ ]:
test_columns = list(test_df.columns)
train_columns = list(train_df.columns)

columns = test_columns + train_columns
print("test col: ",  len(test_columns))
print("train col: ", len(train_columns))
print(len(columns))

In [ ]:
set(columns)
print(len(columns))

In [ ]:
test_df.isna().sum()

In [ ]:
train_df.isna().sum()

In [ ]:
# columns in both dataset is the same
print('true') if list(test_df.columns) == list(train_df.columns) else print('false')

In [ ]:
from sklearn.preprocessing import StandardScaler

test_df = test_df.set_index('id')
train_df = train_df.set_index('id')

train_y = train_df['label']
test_y = test_df['label']

X_train = train_df.drop(columns=['label', 'attack_cat', 'service'])
X_test = test_df.drop(columns=['label', 'attack_cat', 'service'])

num_col = [col for col in X_train.columns if col not in ['state', 'proto']]

scaler = StandardScaler()

X_train[num_col] = scaler.fit_transform(X_train[num_col])
X_test[num_col] = scaler.transform(X_test[num_col])

X_train = pd.get_dummies(X_train, columns=['state', 'proto'], drop_first=True)
X_test = pd.get_dummies(X_test, columns=['state', 'proto'], drop_first=True)

# Ensures that both df shares the same feature columns
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

train_datset = X_train.copy()
train_datset['label'] = train_y

test_dataset = X_test.copy()
test_dataset['label'] = test_y

In [ ]:
print(f'test shape: {test_dataset.shape}')
print(f'train shape: {train_datset.shape}')

In [ ]:
# Converting True/False values to 1/0

train_datset.replace(True, 1, inplace=True)
train_datset.replace(False, 0, inplace=True)

test_dataset.replace(True, 1, inplace=True)
test_dataset.replace(False, 0, inplace=True)

In [ ]:
train_datset.to_csv('dataset/cleaned_dataset/train_dataset.csv')
test_dataset.to_csv('dataset/cleaned_dataset/test_dataset.csv')

In [ ]:
print('true') if list(test_dataset.columns) == list(train_datset.columns) else print('false')

In [ ]:
train_df.columns

In [ ]:
test_df.columns

In [ ]:
print(len(list(test_dataset.columns)))

In [ ]:
# Dropping features based on the results of the feature analysis
# Any feature with a score less then 0.1 was dropped

test_feature_analysis = test_dataset.drop(columns=['sload', 'proto_ospf', 'sjit', 'dur', 
                           'ct_dst_sport_ltm', 'ct_srv_src',
                           'proto_rtp', 'stcpb', 'proto_icmp', 
                           'state_ECO', 'response_body_len',  'ct_src_ltm',
                           'state_URN', 'state_no', 'ct_flw_http_mthd', 'dtcpb', 'state_PAR'])

train_feature_analysis = train_datset.drop(columns=['sload', 'proto_ospf', 'sjit', 'dur', 
                           'ct_dst_sport_ltm', 'ct_srv_src',
                           'proto_rtp', 'stcpb', 'proto_icmp', 
                           'state_ECO', 'response_body_len',  'ct_src_ltm',
                           'state_URN', 'state_no', 'ct_flw_http_mthd', 'dtcpb', 'state_PAR'])


In [ ]:
print(len(test_feature_analysis.columns))
print(len(train_feature_analysis.columns))


In [ ]:
# Confirming that both datasets have the same features
print('true') if list(test_feature_analysis.columns) == list(train_feature_analysis.columns) else print('false')

In [ ]:
test_feature_analysis.to_csv('dataset/cleaned_dataset/test_feature_analysis.csv')
train_feature_analysis.to_csv('dataset/cleaned_dataset/train_feature_analysis.csv')

In [ ]:
# top 25 features from feature analysis

test_top_25 = test_feature_analysis[['dttl', 'dload', 'proto_tcp', 'swin', 'dwin', 'state_INT', 'dpkts', 'proto_unas', 
                                     'dbytes', 'dmean', 'sloss', 'state_FIN', 'sbytes', 'proto_sctp', 
                                     'ct_srv_dst', 'is_sm_ips_ports', 'proto_arp', 'proto_udp', 'ct_state_ttl', 'sttl', 
                                     'ct_src_dport_ltm', 'spkts', 'label']]
train_top_25 = train_feature_analysis[['dttl', 'dload', 'proto_tcp', 'swin', 'dwin', 'state_INT', 'dpkts', 'proto_unas', 
                                     'dbytes', 'dmean', 'sloss', 'state_FIN', 'sbytes', 'proto_sctp', 
                                     'ct_srv_dst', 'is_sm_ips_ports', 'proto_arp', 'proto_udp', 'ct_state_ttl', 'sttl', 
                                     'ct_src_dport_ltm', 'spkts', 'label']]

In [ ]:
test_top_25.to_csv('dataset/cleaned_dataset/test_top_25.csv')
train_top_25.to_csv('dataset/cleaned_dataset/train_top_25.csv')